In [ ]:
# ============================================
# CROSS-DATASET EVALUATION
# Trained on DFDC02 → Tested on DFD01
# ============================================
# Datasets needed:
#   1. preprocessed-dfd01-16 (cross-dataset)
#   2. kaggle_train output (checkpoints from training run)
#      OR preprocessed-dfdc02-16 + separate checkpoints dataset
# ============================================

import subprocess, sys, os

# Step 1: Install dependencies
print('=== Installing dependencies ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy<2', 'scipy<1.17', 'scikit-learn', 'timm', 'facenet-pytorch'],
               check=True)
print('Dependencies installed.')

# Step 2: Clone project
if not os.path.exists('/kaggle/working/project/.git'):
    subprocess.run(['rm', '-rf', '/kaggle/working/project'], check=False)
    subprocess.run(['git', 'clone', 'https://github.com/Jokeera/deepfake-detection.git',
                     '/kaggle/working/project'], check=True)
    print('Project cloned.')
else:
    subprocess.run(['git', '-C', '/kaggle/working/project', 'pull', '--ff-only'],
                   check=True)
    print('Project updated.')

# Step 3: Auto-discover paths
print('\n=== Discovering datasets and checkpoints ===')

# Find DFD01 cross-dataset
cross_dataset = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'real' in dirs and 'fake' in dirs:
        # Check it's DFD01 not DFDC02
        if 'dfd01' in root.lower() or 'DFD01' in root:
            cross_dataset = root
            break
        # If name doesn't help, check content (DFD01 has specific naming)
        real_samples = os.listdir(os.path.join(root, 'real'))[:1]
        if real_samples and '__' in real_samples[0]:
            cross_dataset = root
            break

if cross_dataset is None:
    # Fallback: any preprocessed dataset that's not DFDC02
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'real' in dirs and 'fake' in dirs and 'dfdc02' not in root.lower():
            cross_dataset = root
            break

print(f'Cross-dataset (DFD01): {cross_dataset}')
if cross_dataset:
    real_count = len(os.listdir(os.path.join(cross_dataset, 'real')))
    fake_count = len(os.listdir(os.path.join(cross_dataset, 'fake')))
    print(f'  real={real_count}, fake={fake_count}, total={real_count+fake_count}')

# Find checkpoints (from kaggle_train output or separate dataset)
checkpoints_dir = None
for search_root in ['/kaggle/input']:
    for root, dirs, files in os.walk(search_root):
        if 'best_model.pt' in files and 'metrics.json' in files:
            parent = os.path.dirname(root)
            # Check if parent has multiple experiment dirs
            siblings = [d for d in os.listdir(parent) 
                       if os.path.isdir(os.path.join(parent, d))
                       and os.path.exists(os.path.join(parent, d, 'best_model.pt'))]
            if len(siblings) >= 2:
                checkpoints_dir = parent
                break
    if checkpoints_dir:
        break

print(f'Checkpoints dir: {checkpoints_dir}')
if checkpoints_dir:
    for d in sorted(os.listdir(checkpoints_dir)):
        pt = os.path.join(checkpoints_dir, d, 'best_model.pt')
        if os.path.exists(pt):
            size_mb = os.path.getsize(pt) / 1e6
            print(f'  {d}: {size_mb:.1f} MB')

assert cross_dataset is not None, 'DFD01 dataset not found! Add preprocessed-dfd01-16 as data source.'
assert checkpoints_dir is not None, 'Checkpoints not found! Add kaggle_train output as data source.'

print('\n=== All paths found. Starting cross-eval ===')

In [ ]:
# Step 4: Run cross-dataset evaluation
result = subprocess.run([
    sys.executable, '/kaggle/working/project/kaggle-cross-eval.py',
    '--checkpoints_dir', checkpoints_dir,
    '--cross_dataset', cross_dataset,
    '--output_dir', '/kaggle/working/cross_eval',
    '--device', 'auto',
    '--batch_size', '16',
    '--num_workers', '2',
])
print(f'\nExit code: {result.returncode}')

In [ ]:
# Step 5: Display results
import json

results_path = '/kaggle/working/cross_eval/cross_eval_results.json'
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    
    print('=' * 90)
    print('CROSS-DATASET EVALUATION: Trained on DFDC02 → Tested on DFD01')
    print('=' * 90)
    print(f'{"Model":<35} {"DFDC02 AUC":>12} {"DFD01 AUC":>12} {"Δ AUC":>10} {"DFD01 Acc":>12} {"DFD01 F1":>10}')
    print('-' * 90)
    for r in results:
        m = r['cross_dataset_metrics']
        orig = r['orig_test_auc']
        delta = m['auc'] - orig
        print(f"{r['exp_dir']:<35} {orig:>12.4f} {m['auc']:>12.4f} {delta:>+10.4f} {m['accuracy']:>12.4f} {m['f1']:>10.4f}")
    print('=' * 90)
    
    # Key finding
    full = next((r for r in results if r['model_type'] == 'full'), None)
    spatial = next((r for r in results if r['model_type'] == 'spatial'), None)
    if full and spatial:
        full_drop = full['orig_test_auc'] - full['cross_dataset_metrics']['auc']
        spat_drop = spatial['orig_test_auc'] - spatial['cross_dataset_metrics']['auc']
        print(f'\nA1 Full drop: {full_drop:.4f}')
        print(f'A2 Spatial drop: {spat_drop:.4f}')
        if full_drop < spat_drop:
            print('→ Dual-path обобщает ЛУЧШЕ чем spatial-only (temporal branch помогает)')
        else:
            print('→ Spatial-only обобщает лучше (domain-specific артефакты доминируют)')
else:
    print('Results file not found. Check logs above for errors.')